[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_colab_tutorial_computational_biophysics.ipynb)

# jaxfne: Computational Biophysics Tutorial

**A 4-part journey:** single neurons → small populations → laminar readouts → package tuning metadata.

This tutorial demonstrates the public v0.3 grammar:

```python
cfg = jtfne.Configuration()
cfg = cfg.runtime(...).column(...).cell_types(...).connectivity(...).set_emitter(...).probes(...)
model = jtfne.construct(cfg)
signals = jtfne.simulate(model, ...)
readouts = model.probe(signals, ...)
jtfne.vis.spectrolaminar(signals)
```

All outputs are simulated proxy readouts from a computational scaffold. Numerical values are tutorial-scale outputs.

---

**Parts:**
1. Part 1 — Configured single neuron
2. Part 2 — Configured E/PV population
3. Part 3 — Configured laminar column and package visualizer
4. Part 4 — package objective and tuning report
5. Export — figures, run metadata, and JSON validation

## 1. Learning Objectives

After completing this tutorial, you will understand:

1. **Configuration-first modeling** — how every simulation, including a single neuron, begins with a `jtfne.Configuration` object.
2. **package execution** — how `jtfne.construct` and `jtfne.simulate` replace tutorial-local simulation engines.
3. **Probe/readout extraction** — how `model.probe(...)` exposes spikes, voltage, source proxies, and laminar readouts.
4. **Package visualization** — how `jtfne.vis.spectrolaminar(...)` renders laminar proxy readouts.
5. **Objective/tuning metadata** — how `jtfne.Objective`, `jtfne.random_search`, and `model.tune(...)` report optimization diagnostics for automated optimization.


## 2. Biological / Computational Question

How can the same public jaxfne grammar organize a single neuron, a small population, and a laminar column while preserving clear boundaries between simulated proxy readouts and physical measurement models?


## 3. Mathematical Glossary Flow

### 3.1 Emitter dynamics

**Formal equation:**

$$
\frac{dv_i}{dt} = 0.04 v_i^2 + 5v_i + 140 - u_i + I_{i,\mathrm{Emitter-state-based}}(t)
$$

$$
\frac{du_i}{dt} = a_i(b_i v_i - u_i)
$$

**Terms:**

- $i$ — neuron index.
- $v_i(t)$ — voltage-like state for neuron $i$.
- $u_i(t)$ — recovery state for neuron $i$.
- $a_i, b_i$ — recovery dynamics parameters.
- $I_{drive} input drive used by the reduced emitter.

**Worded equation:**
The voltage-like state changes according to an activation term, a linear term, a recovery feedback term, and a Emitter-state-based input drive. The recovery state follows a slower relaxation process tied to voltage.

**Implementation location:** package emitter kernels are called through `jtfne.construct(...)` and `jtfne.simulate(...)`; the tutorial does not call low-level emitter kernels directly.

**Boundary:** the Emitter-state-based input is a tutorial-scale reduced-emitter drive, not an empirically calibrated current.

### 3.2 Configured recurrent input scaffold

**Formal equation:**

$$
I_i(t) = I_i^{\mathrm{base}}(t) + \sum_j W_{ij}s_j(t)
$$

**Terms:**

- $I_{drive} input to receiving neuron $i$.
- $I_{drive} drive.
- $W_{ij}$ — signed configured connection from sending neuron $j$ to receiving neuron $i$.
- $s_j(t)$ — spike or synaptic activity from neuron $j$.

**Worded equation:**
Each neuron receives its own baseline drive plus a signed sum of activity from other neurons.

**Implementation location:** recurrent metadata is declared through `cfg.connectivity(...)`; the configured model is built with `jtfne.construct(cfg)` and simulated with `jtfne.simulate(model, ...)`.

**Boundary:** this is a Emitter-state-based coupling scaffold for tutorial use, not a calibrated synaptic biophysics model.


In [ ]:
import jaxfne as jtfne
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
import shutil

DURATION_MS = 5000.0
DT_MS = 0.1
SEED = 42
DTYPE = "float32"

FIG_DIR = Path("figures") / "suite_no1_outputs"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Shared plotting helpers: figure layout only; no simulation, source, readout, or objective logic lives here.
def save_png(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"saved: {path} ({path.stat().st_size} bytes)")
    return str(path)

def finite_status(*arrays):
    return all(bool(np.all(np.isfinite(np.asarray(a)))) for a in arrays if a is not None)

def population_rate_hz(spikes, dt_ms):
    spikes = np.asarray(spikes)
    if spikes.size == 0:
        return 0.0
    return float(spikes.mean() * (1000.0 / dt_ms))

run_metadata = {
    "truth_mode": "truth_safe_unverified",
    "scope_status": "computational_scaffold",
    "readout_status": "simulated_proxy",
    "field_solver_status": "laminar_proxy_no_pde",
    "physical_amplitude_allowed": False,
    "duration_ms": DURATION_MS,
    "dt_ms": DT_MS,
    "dtype": DTYPE,
    "seed": SEED,
}
json.dumps(run_metadata, allow_nan=False)
print(json.dumps(run_metadata, indent=2))


## 4. Canonical Import

The notebook imports jaxfne only as:

```python
import jaxfne as jtfne
```

All simulations below use configuration objects, constructed models, package simulation, package probes, and package visualization where available.


## 5. Configuration Block — Part 1: Single Neuron

A single neuron is still a configured jaxfne model.


In [ ]:
cfg_single = jtfne.Configuration()
cfg_single = cfg_single.runtime(seed=SEED, dtype=DTYPE, duration_ms=DURATION_MS, dt_ms=DT_MS)
cfg_single = cfg_single.column("single_neuron", layers=["L2/3"], n=1)
cfg_single = cfg_single.cell_types({"E": 1.0})
cfg_single = cfg_single.connectivity(kind="none")
cfg_single = cfg_single.set_emitter("izhikevich", "cortical_eig")
cfg_single = cfg_single.probes(["spikes", "V_m", "source", "LFP"], n_contacts=4)

print("single cfg valid:", cfg_single.validate()["valid"])
print("single probes:", cfg_single.probes)


## 6. Simulation Block — Part 1: Single Neuron


In [ ]:
model_single = jtfne.construct(cfg_single)
signals_single = jtfne.simulate(model_single, duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED)
readouts_single = model_single.probe(signals_single, ["spikes", "V_m", "source", "LFP"])

single_v = np.asarray(readouts_single["V_m"])[:, 0]
single_spikes = np.asarray(readouts_single["spikes"])[:, 0]
single_source = np.asarray(readouts_single["sources"])[:, 0]
single_t = np.asarray(signals_single.time_ms)
single_rate = population_rate_hz(single_spikes[:, None], DT_MS)

print(f"single shape: {signals_single.V_m.shape}")
print(f"single spikes: {int(single_spikes.sum())}")
print(f"single mean rate: {single_rate:.2f} Hz")
print("finite:", finite_status(single_v, single_spikes, single_source))


## 7. Probe / Readout Block — Part 1 Figures


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(single_t, single_v, lw=1.0)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Voltage-like state")
ax.set_title("Figure 01 — Configured single-neuron voltage-like trajectory")
save_png(fig, "01_configured_single_voltage")

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(single_t, single_source, lw=1.0)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Source proxy")
ax.set_title("Figure 02 — Configured single-neuron source proxy")
save_png(fig, "02_configured_single_source_proxy")


## 8. Configuration and Simulation — Part 2: Small E/PV Population

The population section uses the same public grammar, with a declared E/PV split and configured connectivity metadata.


In [ ]:
cfg_pop = jtfne.Configuration()
cfg_pop = cfg_pop.runtime(seed=SEED + 1, dtype=DTYPE, duration_ms=DURATION_MS, dt_ms=DT_MS)
cfg_pop = cfg_pop.column("small_epv_population", layers=["L2/3"], n=16)
cfg_pop = cfg_pop.cell_types({"E": 0.75, "PV": 0.25})
cfg_pop = cfg_pop.connectivity(kind="dense_signed_ei_metadata", e_to_all=0.08, i_to_all=-0.12)
cfg_pop = cfg_pop.set_emitter("izhikevich", "cortical_eig")
cfg_pop = cfg_pop.probes(["spikes", "V_m", "source", "LFP"], n_contacts=8)

model_pop = jtfne.construct(cfg_pop)
signals_pop = jtfne.simulate(model_pop, duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED + 1)
readouts_pop = model_pop.probe(signals_pop, ["spikes", "V_m", "source", "LFP"])

pop_t = np.asarray(signals_pop.time_ms)
pop_spikes = np.asarray(readouts_pop["spikes"])
pop_v = np.asarray(readouts_pop["V_m"])
pop_rate = population_rate_hz(pop_spikes, DT_MS)

print("population cfg valid:", cfg_pop.validate()["valid"])
print(f"population shape: {pop_v.shape}")
print(f"population mean rate: {pop_rate:.2f} Hz")
print("finite:", finite_status(pop_v, pop_spikes))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for neuron_id in range(pop_spikes.shape[1]):
    spike_times = pop_t[pop_spikes[:, neuron_id] > 0.5]
    ax.scatter(spike_times, np.full_like(spike_times, neuron_id), s=3)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Neuron index")
ax.set_title("Figure 03 — Configured E/PV population raster")
save_png(fig, "03_configured_population_raster")

bin_ms = 25.0
bin_edges = np.arange(0.0, DURATION_MS + bin_ms, bin_ms)
rates = []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (pop_t >= lo) & (pop_t < hi)
    rates.append(float(pop_spikes[mask].mean() * (1000.0 / DT_MS)))
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(0.5 * (bin_edges[:-1] + bin_edges[1:]), rates, lw=1.5)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Mean rate (Hz)")
ax.set_title("Figure 04 — Configured population rate summary")
save_png(fig, "04_configured_population_rate")

# Connectivity visualization is derived from the constructed model metadata/parameters.
W = np.asarray(model_pop.params["emitter"].W)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(W, aspect="auto")
ax.set_xlabel("Sending neuron")
ax.set_ylabel("Receiving neuron")
ax.set_title("Figure 05 — Constructed connectivity matrix")
fig.colorbar(im, ax=ax, fraction=0.046)
save_png(fig, "05_constructed_connectivity_matrix")


## 9. Configuration and Simulation — Part 3: Laminar Column

The laminar section is also built from a `Configuration`. Readouts are extracted through `model.probe(...)`, and the package visualizer renders the spectrolaminar profile.


In [ ]:
cfg_column = jtfne.Configuration()
cfg_column = cfg_column.runtime(seed=SEED + 2, dtype=DTYPE, duration_ms=DURATION_MS, dt_ms=DT_MS)
cfg_column = cfg_column.column("laminar_column", layers=["L2/3", "L4", "L5", "L6"], n=48)
cfg_column = cfg_column.cell_types({"E": 0.75, "PV": 0.12, "SST": 0.08, "VIP": 0.05})
cfg_column = cfg_column.connectivity(kind="laminar_signed_metadata", recurrent=True)
cfg_column = cfg_column.set_emitter("izhikevich", "cortical_eig")
cfg_column = cfg_column.probes(["spikes", "V_m", "source", "LFP", "CSD"], n_contacts=16)

model_column = jtfne.construct(cfg_column)
signals_column = jtfne.simulate(model_column, duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED + 2)
readouts_column = model_column.probe(signals_column, ["spikes", "V_m", "source", "LFP", "CSD"])

column_spikes = np.asarray(readouts_column["spikes"])
column_lfp = np.asarray(readouts_column["lfp_proxy"])
column_csd = np.asarray(readouts_column["csd_proxy"])
column_rate = population_rate_hz(column_spikes, DT_MS)

print("column cfg valid:", cfg_column.validate()["valid"])
print(f"column spikes: {column_spikes.shape}")
print(f"column LFP-proxy: {column_lfp.shape}")
print(f"column CSD-proxy: {column_csd.shape}")
print(f"column mean rate: {column_rate:.2f} Hz")
print("finite:", finite_status(column_spikes, column_lfp, column_csd))


In [ ]:
# Built-in visualization call.
fig = jtfne.vis.spectrolaminar(signals_column, figsize=(13, 4))
save_png(fig, "06_jtfne_vis_spectrolaminar")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(np.asarray(signals_column.time_ms), column_lfp[:, :4], lw=0.8)
axes[0].set_title("LFP-proxy channels")
axes[0].set_xlabel("Time (ms)")
axes[0].set_ylabel("Proxy units")
axes[1].imshow(column_csd.T, aspect="auto", origin="upper")
axes[1].set_title("CSD-proxy heatmap")
axes[1].set_xlabel("Time index")
axes[1].set_ylabel("Contact")
fig.suptitle("Figure 07 — Configured laminar readout summary")
save_png(fig, "07_configured_laminar_readout_summary")


## 10. package Objective and Tuning Metadata — Part 4

This section uses package objective and optimizer objects. It does not define a tutorial-local loss function or random-search loop.


In [ ]:
objective = jtfne.objective()
objective = objective.loss("rate_target", metric="spike_rate_hz_mean", target=10.0, weight=1.0)
objective = objective.gate("finite_rate_window", metric="spike_rate_hz_mean", threshold=[0.0, 100.0], criterion="in_range")

optimizer = jtfne.random_search(metadata={"tutorial_section": "suite_no1_part4"})
short_sim = jtfne.Simulation(duration_ms=300.0, dt_ms=DT_MS, seed=SEED + 3)
model_tuned, tuning_report_raw = model_column.tune(
    objective,
    optimizer=optimizer,
    steps=3,
    seed=SEED + 3,
    simulation=short_sim,
    parameter="source_scale",
    bounds=(0.5, 1.5),
)

# Display a public-facing subset only.
tuning_report = {
    "tuning_status": tuning_report_raw.get("tuning_status"),
    "acceptance_decision": tuning_report_raw.get("acceptance_decision"),
    "steps_requested": tuning_report_raw.get("steps_requested"),
    "best_parameter_value": tuning_report_raw.get("best_parameter_value"),
    "best_score": tuning_report_raw.get("best_score"),
    "scope_status": "computational_scaffold",
}
json.dumps(tuning_report, allow_nan=False)
print(json.dumps(tuning_report, indent=2))

fig, ax = plt.subplots(figsize=(7, 3))
history = tuning_report_raw.get("candidate_history", [])
steps = [h["step"] for h in history]
scores = [np.nan if h.get("score") is None else h.get("score") for h in history]
ax.plot(steps, scores, marker="o")
ax.set_xlabel("Candidate step")
ax.set_ylabel("Objective score")
ax.set_title("Figure 08 — Package tuning report summary")
save_png(fig, "08_package_tuning_report_summary")


## 11. Run Metadata and Scope Metadata

The notebook records strict JSON-safe metadata for the run. These metadata describe tutorial scope, runtime settings, and proxy readout status.


In [ ]:
figure_files = sorted(str(p) for p in FIG_DIR.glob("*.png"))
validation_summary = {
    **run_metadata,
    "configured_models": ["cfg_single", "cfg_pop", "cfg_column"],
    "public_api_path": [
        "jtfne.Configuration",
        "cfg.runtime",
        "cfg.column",
        "cfg.cell_types",
        "cfg.connectivity",
        "cfg.set_emitter",
        "cfg.probes",
        "jtfne.construct",
        "jtfne.simulate",
        "model.probe",
        "jtfne.vis.spectrolaminar",
        "model.tune",
    ],
    "n_figures": len(figure_files),
    "figures": figure_files,
    "finite_outputs": bool(finite_status(single_v, pop_v, column_lfp, column_csd)),
    "single_rate_hz": round(single_rate, 6),
    "population_rate_hz": round(pop_rate, 6),
    "column_rate_hz": round(column_rate, 6),
}
json.dumps(validation_summary, allow_nan=False)
Path("suite_no1_run_metadata.json").write_text(json.dumps(validation_summary, indent=2), encoding="utf-8")
print(json.dumps({k: validation_summary[k] for k in ["n_figures", "finite_outputs", "single_rate_hz", "population_rate_hz", "column_rate_hz"]}, indent=2))


## 12. Interpretation

The tutorial demonstrates the intended public flow:

1. The user declares a configuration.
2. jaxfne constructs a model from that configuration.
3. jaxfne simulates the model.
4. The model extracts probe/readout arrays.
5. jaxfne visualization renders the laminar readout.
6. jaxfne objective and optimizer objects produce tuning metadata.

The output patterns are useful for learning and debugging the source/field/probe scaffold. They should be interpreted as simulated proxy readouts from the declared configuration.

## 13. Failure Modes

Common failure modes:

- **No configured model:** calling low-level kernels directly bypasses the public tutorial grammar.
- **Unstable rate:** excessive Emitter-state-based drive or coupling can produce too much synchrony or silence.
- **Missing plotting dependency:** visualization requires the optional visualization dependency, while core import remains lightweight.
- **Metadata drift:** generated outputs should stay separate from stable docs assets unless intentionally refreshed.
- **Over-interpretation:** proxy readouts should not be read as calibrated sensor measurements.


## 14. Exercises

1. Change the E/PV split in `cfg_pop.cell_types(...)` and rerun the population raster.
2. Increase the number of laminar contacts in `cfg_column.probes(..., n_contacts=...)` and inspect the spectrolaminar figure.
3. Switch `optimizer = jtfne.random_search(...)` to `jtfne.gsdr(...)` and compare the tuning metadata.
4. Add a second configured column and inspect how the model summary changes.
5. Export the figure directory and the JSON metadata for a reproducibility check.


## 15. What This Tutorial Covers / Does Not Cover

**Covers:**

- configuration-first single-neuron, population, and laminar-column examples;
- package construction and simulation;
- probe/readout extraction;
- package spectrolaminar visualization;
- package objective and tuning metadata.

**Does not cover:**

- calibrated physical amplitudes;
- subject-specific sensor geometry;
- empirical validation;
- detailed compartmental morphology;
- mechanistic proof of a biological circuit.

*Suite No. 1  notebook | truth_safe_unverified | computational_scaffold | simulated/proxy readouts.*


In [ ]:
archive_base = "suite_no1_outputs"
archive_path = shutil.make_archive(archive_base, "zip", FIG_DIR.parent, FIG_DIR.name)